# HistoVision — обучение сегментационной модели на BCSS (Colab GPU)

Прогоняется через официальное расширение **Colab для VS Code** (Kernel → Select Kernel → Colab → New Colab Server → GPU/T4), либо напрямую в браузере на colab.research.google.com.

**Важно:** в списке ядер VS Code может быть несколько вариантов (в т.ч. Julia, если он у вас установлен локально) — убедитесь, что выбран именно **Python 3** / Colab-рантайм, а не Julia. Признак, что ядро не то: `!nvidia-smi` в первой ячейке падает с `UndefVarError` — это Julia, а не Python, пытается выполнить строку как код.

**Если ячейка 1 показывает `nvidia-smi: command not found` / `CUDA available: False`, хотя вы выбирали GPU/T4** — значит подключился CPU-рантайм. "Restart" в VS Code перезапускает только Python-процесс, а не саму Colab-машину, поэтому если раньше уже был создан CPU-сеанс, "Restart" к нему и переподключит. Решение: через кнопку выбора ядра явно создайте **новый** Colab-сеанс (New Colab Server) с GPU, а не переиспользуйте существующий.

Код тянется из GitHub-репозитория, ветка `feature/segmentation` — URL уже прописан во второй code-ячейке. Ячейка 2 сама разбирается, клонировать репозиторий с нуля или обновить существующий (`git pull`) — безопасно перезапускать.

**Про ячейку 4 (скачивание BCSS):** Google Drive иногда блокирует скачивание отдельных файлов из-за дневной квоты на количество обращений (популярный публичный датасет). Это не баг — скрипт пропускает такие файлы и продолжает с остальными, в конце пишет, сколько пар скачалось и сколько пропущено. Уже скачанные файлы при повторном запуске не перекачиваются.

**Что делает этот ноутбук:**
1. Проверяет GPU-рантайм.
2. Клонирует репозиторий и ставит зависимости.
3. Скачивает и готовит датасет BCSS (`prepare_bcss.py download`).
4. Обучает DeepLabV3+/ResNet-18 (`train.py`) на GPU.
5. Сохраняет чекпоинт и скачивает его на локальную машину — для дальнейшей интеграции в `src/inference/`.

In [ ]:
# 1. Проверка GPU
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (нет GPU-рантайма — выберите GPU в Select Kernel)")

In [ ]:
# 2. Клонирование репозитория (или git pull, если /content уже содержит клон
# с прошлого запуска — VS Code "Restart" перезапускает только Python-процесс,
# не саму Colab-машину, так что файлы в /content переживают рестарт).
# git pull трогает только код из репозитория; data/bcss — вне git
# (gitignored), поэтому уже скачанные файлы при квоте Google Drive не
# теряются между повторными запусками.
import os

REPO_URL = "https://github.com/soon00sad/histo_learn.git"
BRANCH = "feature/segmentation"
REPO_DIR = "/content/HistoVision"

if os.path.isdir(f"{REPO_DIR}/.git"):
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
# 3. Зависимости. torch/torchvision и большинство ML-пакетов в Colab уже стоят —
# их не трогаем. pydantic/PyYAML НЕ пинуем здесь (в отличие от requirements.txt
# основного проекта) — жёсткая версия конфликтует с уже установленными в Colab
# пакетами (google-genai/google-adk/albumentations/fastapi требуют более новый
# pydantic); наш src/utils/bcss_classes.py и config.py не используют ничего
# version-specific, так что более новый pydantic/PyYAML тут безопасен.
!pip install -q segmentation-models-pytorch==0.5.0 gdown==6.1.0

In [ ]:
# 4. Подготовка BCSS. Полный датасет с Google Drive — может быть медленно/большим,
# это нормально на GPU-машине с нормальной сетью (в отличие от dev-машины автора).
# --limit можно убрать для полного датасета, либо оставить для первого прогона.
!python -m src.training.prepare_bcss download --out data/bcss --limit 60

In [ ]:
# 5. Обучение. Подберите --epochs/--batch-size под реальный размер датасета и GPU.
!python -m src.training.train \
    --data-dir data/bcss \
    --out models/segmentation.pth \
    --encoder resnet18 \
    --epochs 40 \
    --batch-size 16 \
    --crop-size 256 \
    --device cuda

In [ ]:
# 6. Скачать чекпоинт локально (для интеграции в src/inference/ на dev-машине).
# В браузерном Colab сработает files.download; из VS Code проще открыть файл
# в проводнике Colab-рантайма и скачать вручную, либо загрузить в Google Drive:
try:
    from google.colab import files
    files.download("models/segmentation.pth")
except Exception as e:
    print("files.download недоступен в этом окружении (", e, ") — скачайте models/segmentation.pth вручную.")